# Семинар 6. Ансамбли: от бэггинга до CatBoost

В этом семинаре мы разберем:
- Bagging (bootstrap aggregating)
- Random Forest (OOB score, feature importance)
- Boosting (AdaBoost, Gradient Boosting)
- CatBoost deep dive (ordered boosting, categorical features, early stopping, SHAP)
- CatBoost vs XGBoost vs LightGBM
- Stacking

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q catboost xgboost lightgbm shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings("ignore")

from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

np.random.seed(42)

In [ ]:
def plot_decision_boundary(clf, X, y, ax, title):
    eps = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - eps, X[:, 0].max() + eps, 300),
        np.linspace(X[:, 1].min() - eps, X[:, 1].max() + eps, 300),
    )
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cmap_bg = ListedColormap(['#FFAAAA', '#AAAAFF'])
    ax.pcolormesh(xx, yy, Z, cmap=cmap_bg, alpha=0.4)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolors='k', s=20)
    ax.set_title(title)
    ax.grid(alpha=0.2)

## 1. Bagging

**Bootstrap Aggregating (Bagging)** - обучаем N моделей на случайных подвыборках (с возвратом) и усредняем их предсказания. Это снижает дисперсию (variance) модели.

Каждая подвыборка содержит ~63.2% уникальных объектов (остальные - дубликаты). Оставшиеся ~36.8% объектов можно использовать для оценки качества (OOB).

In [ ]:
X_moon, y_moon = make_moons(n_samples=500, noise=0.3, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_moon, y_moon, test_size=0.3, random_state=42)

In [ ]:
tree = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42,
).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_decision_boundary(tree, X_te, y_te, axes[0],
    f'Decision Tree (acc={tree.score(X_te, y_te):.3f})')
plot_decision_boundary(bag, X_te, y_te, axes[1],
    f'Bagging, 100 trees (acc={bag.score(X_te, y_te):.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# Влияние количества деревьев
n_range = [1, 5, 10, 25, 50, 100, 200]
scores = []
for n in n_range:
    b = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=n, random_state=42)
    b.fit(X_tr, y_tr)
    scores.append(b.score(X_te, y_te))

plt.figure(figsize=(10, 5))
plt.plot(n_range, scores, 'o-')
plt.xlabel('n_estimators')
plt.ylabel('Test accuracy')
plt.title('Bagging: effect of n_estimators')
plt.grid(True)
plt.show()

## 2. Random Forest

Random Forest = Bagging + случайный выбор подмножества признаков в каждом узле. Это дополнительно декоррелирует деревья и улучшает ансамбль.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf.fit(X_tr, y_tr)

print(f'Test accuracy: {rf.score(X_te, y_te):.4f}')
print(f'OOB score:     {rf.oob_score_:.4f}  (free validation!)')

fig, ax = plt.subplots(figsize=(8, 6))
plot_decision_boundary(rf, X_te, y_te, ax,
    f'Random Forest (acc={rf.score(X_te, y_te):.3f})')
plt.show()

### Feature importance: MDI vs Permutation

In [ ]:
# Более интересные данные для feature importance
X_fi, y_fi = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, random_state=42,
)
feat_names = [f'feat_{i}' for i in range(10)]
X_fi_tr, X_fi_te, y_fi_tr, y_fi_te = train_test_split(X_fi, y_fi, test_size=0.3, random_state=42)

rf_fi = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_fi_tr, y_fi_tr)

# MDI (mean decrease impurity) - built-in
mdi = rf_fi.feature_importances_

# Permutation importance
perm = permutation_importance(rf_fi, X_fi_te, y_fi_te, n_repeats=10, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
idx_mdi = np.argsort(mdi)
axes[0].barh(range(10), mdi[idx_mdi])
axes[0].set_yticks(range(10), [feat_names[i] for i in idx_mdi])
axes[0].set_title('MDI (mean decrease impurity)')
axes[0].grid(True, axis='x')

idx_perm = np.argsort(perm.importances_mean)
axes[1].barh(range(10), perm.importances_mean[idx_perm])
axes[1].set_yticks(range(10), [feat_names[i] for i in idx_perm])
axes[1].set_title('Permutation Importance')
axes[1].grid(True, axis='x')

plt.suptitle(f'Random Forest feature importance (test acc={rf_fi.score(X_fi_te, y_fi_te):.3f})', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Boosting

В отличие от бэггинга (параллельное обучение), бустинг обучает модели последовательно: каждая следующая модель исправляет ошибки предыдущих.

**AdaBoost**: увеличивает вес неправильно классифицированных объектов.

**Gradient Boosting**: каждая следующая модель обучается предсказывать остатки (градиент лосса) ансамбля.

In [ ]:
ada = AdaBoostClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
plot_decision_boundary(rf, X_te, y_te, axes[0],
    f'Random Forest (acc={rf.score(X_te, y_te):.3f})')
plot_decision_boundary(ada, X_te, y_te, axes[1],
    f'AdaBoost (acc={ada.score(X_te, y_te):.3f})')
plot_decision_boundary(gb, X_te, y_te, axes[2],
    f'Gradient Boosting (acc={gb.score(X_te, y_te):.3f})')
plt.suptitle('Bagging (RF) vs Boosting (AdaBoost, GradientBoosting)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. CatBoost deep dive

### 4.1 Ordered boosting

Классический градиентный бустинг использует таргет для вычисления остатков, что может приводить к утечке информации. CatBoost использует ordered boosting: для каждого объекта остатки считаются только по объектам, которые идут раньше него в случайной перестановке.

### 4.2 Symmetric (oblivious) trees

CatBoost строит симметричные деревья: на каждом уровне используется одно и то же разбиение для всех узлов. Это быстрее при инференсе и дает неявную регуляризацию (дерево не может быть слишком сложным).

### 4.3 Категориальные признаки

CatBoost умеет работать с категориальными признаками напрямую: использует ordered target encoding (среднее таргета по объектам, идущим раньше в перестановке). Не нужно делать OHE или Label Encoding вручную.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/a-milenkin/Competitive_Data_Science/main/data"
car_train = pd.read_csv(f"{DATA_URL}/car_train.csv")
print(car_train.shape)
car_train.head()

In [ ]:
cat_cols = ['model', 'car_type', 'fuel_type']
target = 'target_class'
features = ['car_rating', 'year_to_start', 'riders', 'year_to_work'] + cat_cols

X_car = car_train[features]
y_car = car_train[target]
X_car_tr, X_car_te, y_car_tr, y_car_te = train_test_split(
    X_car, y_car, test_size=0.2, random_state=42,
)

# CatBoost с указанием категориальных признаков
cb_cat = CatBoostClassifier(
    iterations=200, depth=6, cat_features=cat_cols,
    verbose=False, random_state=42,
)
cb_cat.fit(X_car_tr, y_car_tr)
print(f'CatBoost (cat_features): {cb_cat.score(X_car_te, y_car_te):.4f}')

### 4.4 Training: eval_set + early stopping

CatBoost поддерживает мониторинг качества на валидационной выборке во время обучения. `early_stopping_rounds` останавливает обучение, если качество не улучшается.

In [ ]:
cb_es = CatBoostClassifier(
    iterations=1000, depth=6, learning_rate=0.05,
    cat_features=cat_cols, verbose=False, random_state=42,
)
cb_es.fit(
    X_car_tr, y_car_tr,
    eval_set=(X_car_te, y_car_te),
    early_stopping_rounds=50,
)
print(f'Best iteration: {cb_es.best_iteration_}')
print(f'Test accuracy: {cb_es.score(X_car_te, y_car_te):.4f}')

# Plot learning curves
evals = cb_es.evals_result_
plt.figure(figsize=(10, 5))
plt.plot(evals['learn']['MultiClass'], label='Train', alpha=0.7)
plt.plot(evals['validation']['MultiClass'], label='Validation', alpha=0.7)
plt.axvline(x=cb_es.best_iteration_, color='r', linestyle='--', label=f'Best iter={cb_es.best_iteration_}')
plt.xlabel('Iteration')
plt.ylabel('MultiClass Loss')
plt.title('CatBoost: early stopping')
plt.legend()
plt.grid(True)
plt.show()

### 4.5 Feature importance

CatBoost поддерживает несколько типов feature importance:
- **PredictionValuesChange** (default): насколько меняется предсказание при разбиении по признаку
- **LossFunctionChange**: насколько меняется лосс
- **ShapValues**: на основе SHAP (самый точный, но медленный)

In [ ]:
fi_pvc = cb_es.get_feature_importance(type='PredictionValuesChange', prettified=True)

train_pool = Pool(X_car_tr, label=y_car_tr, cat_features=cat_cols)
fi_lfc = cb_es.get_feature_importance(data=train_pool, type='LossFunctionChange', prettified=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fi_df, title in zip(axes, [fi_pvc, fi_lfc],
    ['PredictionValuesChange', 'LossFunctionChange']):
    ax.barh(fi_df['Feature Id'], fi_df['Importances'])
    ax.set_title(title)
    ax.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# ShapValues для CatBoost:
# fi_shap = cb_es.get_feature_importance(data=train_pool, type='ShapValues')
# Для multiclass результат 3D: (n_classes, n_samples, n_features+1).
# Подробнее про SHAP - в семинаре 2 (Feature Selection).
print("Также доступен тип ShapValues - самый точный, но медленный.")

### 4.6 CatBoost vs XGBoost vs LightGBM

In [ ]:
# Данные для сравнения (числовые, без категорий - для честного сравнения)
X_cmp, y_cmp = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    n_classes=3, random_state=42,
)
X_cmp_tr, X_cmp_te, y_cmp_tr, y_cmp_te = train_test_split(
    X_cmp, y_cmp, test_size=0.2, random_state=42,
)

results = {}

for name, clf in [
    ('CatBoost', CatBoostClassifier(iterations=200, verbose=False, random_state=42)),
    ('XGBoost', XGBClassifier(n_estimators=200, verbosity=0, random_state=42)),
    ('LightGBM', LGBMClassifier(n_estimators=200, verbose=-1, random_state=42)),
]:
    t0 = time.time()
    clf.fit(X_cmp_tr, y_cmp_tr)
    train_time = time.time() - t0
    acc = clf.score(X_cmp_te, y_cmp_te)
    results[name] = {'accuracy': acc, 'time': train_time}
    print(f'{name:10s}: accuracy={acc:.4f}, time={train_time:.2f}s')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = list(results.keys())
axes[0].bar(names, [r['accuracy'] for r in results.values()])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy comparison')
axes[0].grid(True, axis='y')

axes[1].bar(names, [r['time'] for r in results.values()])
axes[1].set_ylabel('Training time (s)')
axes[1].set_title('Training time comparison')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

## 5. Stacking

Stacking использует предсказания нескольких моделей как признаки для мета-модели. Идея: разные модели делают разные ошибки, мета-модель учится комбинировать их оптимально.

In [ ]:
stack = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3,
)

models = {
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42),
    'Stacking (RF+GB)': stack,
}

for name, clf in models.items():
    clf.fit(X_cmp_tr, y_cmp_tr)
    acc = clf.score(X_cmp_te, y_cmp_te)
    print(f'{name:25s}: {acc:.4f}')

## Итоги

| Метод | Тип | Ключевая идея | Когда использовать |
|---|---|---|---|
| Bagging | Параллельный | Усреднение по bootstrap-выборкам | Снижение дисперсии |
| Random Forest | Параллельный | Bagging + случайные признаки | Дефолтный baseline |
| AdaBoost | Последовательный | Перевзвешивание объектов | Исторический интерес |
| Gradient Boosting | Последовательный | Обучение на остатках | Высокое качество |
| CatBoost | Последовательный | Ordered boosting + категории | Данные с категориями |
| XGBoost | Последовательный | Regularized boosting | Kaggle classic |
| LightGBM | Последовательный | Leaf-wise growth, GOSS | Большие данные, скорость |
| Stacking | Мета-обучение | Комбинация моделей | Максимальное качество |